In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os
import pickle

In [2]:
#1.READ DATA.
data = pd.read_pickle("../../data/processed/data_processed.pkl")
train_df = pd.read_pickle("../../data/processed/train.pkl")
val_df = pd.read_pickle("../../data/processed/val.pkl")
test_df = pd.read_pickle("../../data/processed/test.pkl")

In [3]:
#2.FUNCTION.

def safe_corr(a, b, eps=1e-8):
    #eps ~ 0, tránh sài số 0.
    #Tính tương quan giữa các trục acc x-y-z và gyr x-y-z
    #Nếu kết quả ra NaN thì quy về 0 vì nó không có ý nghĩa khi so với trục khác.
    if np.std(a) < eps or np.std(b) < eps:
        return 0.0 #float
    return np.corrcoef(a, b)[0, 1] # return corr(a,b)


def extract_window_features(df, window_size=10):
    #mỗi window là 10 (Mốc là 2s để đánh giá.).
    # Chỉ giữ những feature giúp phân biệt activity tốt nhất
    # -> tránh overfitting

    #Ds cột cần xử lý.
    sensor_cols = [
        'acc_x', 'acc_y', 'acc_z',
        'gyr_x', 'gyr_y', 'gyr_z'
    ]

    features_list = []

    #Duyệt theo từng người, set.
    for (user_id, set_id), group in df.groupby(['user_id', 'set']):
        n_windows = len(group) // window_size 
        #Duyệt qua từng window trong 1 set.
        for i in range(n_windows):
            start = i * window_size
            end = start + window_size
            window = group.iloc[start:end]

            #Ko đủ mẫu thì bỏ qua -> tránh lỗi tính toán.
            if len(window) < window_size:
                continue
            
            #Dict.
            window_features = {
                'user_id': user_id,
                'set': set_id,
                'label': window['label'].iloc[0],
                'weight': window['weight'].iloc[0],
                'height': window['height'].iloc[0],
                'age': window['age'].iloc[0],
                'gender': window['gender'].iloc[0]
            }

            #Duyệt qua từng cột -> trích suất đặc trưng (4 cột đại diện) -> tránh overfitting.
            #Đánh giá riêng lẻ trên x,y,z -> Chuyển động theo hướng nào.
            for col in sensor_cols:
                values = window[col].values

                window_features[f'{col}_mean'] = np.mean(values) # vị trí trung tâm. -> giúp phân biệt hướng.
                window_features[f'{col}_std'] = np.std(values) # độ biến động  -> giúp phân biệt tĩnh/ động
                window_features[f'{col}_rms'] = np.sqrt(np.mean(values ** 2)) #cường độ thật -> đo cường độ.
                window_features[f'{col}_energy'] = np.mean(values ** 2) #công suất trong 2s.

            # Gộp 3 trục -> giúp ko phụ thuộc vào hướng đặt điện thoại.
            acc_mag = np.sqrt(
                window['acc_x']**2 +
                window['acc_y']**2 +
                window['acc_z']**2
            )

            gyr_mag = np.sqrt(
                window['gyr_x']**2 +
                window['gyr_y']**2 +
                window['gyr_z']**2
            )

            # Đánh giá magnitude -> Chuyển động mạnh như nào? (Tránh TH xoay điện thoại).
            window_features['acc_mag_mean'] = np.mean(acc_mag)
            window_features['acc_mag_std'] = np.std(acc_mag)
            window_features['acc_mag_rms'] = np.sqrt(np.mean(acc_mag ** 2))

            window_features['gyr_mag_mean'] = np.mean(gyr_mag)
            window_features['gyr_mag_std'] = np.std(gyr_mag)
            window_features['gyr_mag_rms'] = np.sqrt(np.mean(gyr_mag ** 2))

            features_list.append(window_features) 
    #Những feature trên giúp dự đoán xem trong 2s chuyển động đó là gì (chạy,đi bộ,...)
    return pd.DataFrame(features_list)
#OUTCOME: Hàm safe_corr() là để tính tương quan giữa các trục x-y-z của acc và gyr(chỉ 2 đơn vị này).
#         Hàm extract_window_features, lấy 10 mẫu cho mỗi lần -> giảm tgian tính toán feature -> sài std, mean, RSM, energy:
#         Tính 2 lần: 1 lần là cho từng trục riêng lẻ, 1 lần là tổng quát cả 3 trục (ko phụ thuộc vào hướng như riêng lẻ) giúp phân biệt rõ hơn.

In [4]:
#EXTRACT FEATURE (DATA TRANSFORM).
print("\nExtracting features from windows...")
WINDOW_SIZE = 10  # 10 samples × 200ms = 2 seconds window

features_data = extract_window_features(data, window_size=WINDOW_SIZE)
features_train = extract_window_features(train_df, window_size=WINDOW_SIZE)
features_val = extract_window_features(val_df, window_size=WINDOW_SIZE)
features_test = extract_window_features(test_df, window_size=WINDOW_SIZE)


Extracting features from windows...


In [5]:
features_data

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1,1,0,102,188,46,1,0.715572,0.136997,0.728568,...,0.020860,0.262813,0.263640,0.069506,1.052953,0.121381,1.059927,1.301077,0.771064,1.512396
1,1,1,0,102,188,46,1,0.718558,0.198869,0.745570,...,0.374454,0.494895,0.620594,0.385137,1.012239,0.227629,1.037517,1.850563,0.793093,2.013350
2,1,1,0,102,188,46,1,0.715490,0.161866,0.733571,...,0.083601,0.400974,0.409597,0.167769,1.019074,0.184435,1.035629,1.538961,0.785672,1.727913
3,1,1,0,102,188,46,1,0.788775,0.121680,0.798105,...,0.066443,0.303860,0.311039,0.096746,1.083907,0.157921,1.095351,1.401988,0.910345,1.671616
4,1,1,0,102,188,46,1,0.752540,0.167665,0.770991,...,0.218401,0.621696,0.658942,0.434205,1.008821,0.175300,1.023938,1.738690,0.875107,1.946499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,24,353,2,74,173,18,0,-0.192117,0.232299,0.301449,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
13984,24,353,2,74,173,18,0,-0.236467,0.225630,0.326842,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
13985,24,353,2,74,173,18,0,-0.209301,0.203869,0.292180,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
13986,24,353,2,74,173,18,0,-0.247126,0.215143,0.327655,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [6]:
features_train

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1,1,0,102,188,46,1,0.715572,0.136997,0.728568,...,0.020860,0.262813,0.263640,0.069506,1.052953,0.121381,1.059927,1.301077,0.771064,1.512396
1,1,1,0,102,188,46,1,0.718558,0.198869,0.745570,...,0.374454,0.494895,0.620594,0.385137,1.012239,0.227629,1.037517,1.850563,0.793093,2.013350
2,1,1,0,102,188,46,1,0.715490,0.161866,0.733571,...,0.083601,0.400974,0.409597,0.167769,1.019074,0.184435,1.035629,1.538961,0.785672,1.727913
3,1,1,0,102,188,46,1,0.788775,0.121680,0.798105,...,0.066443,0.303860,0.311039,0.096746,1.083907,0.157921,1.095351,1.401988,0.910345,1.671616
4,1,1,0,102,188,46,1,0.752540,0.167665,0.770991,...,0.218401,0.621696,0.658942,0.434205,1.008821,0.175300,1.023938,1.738690,0.875107,1.946499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10440,18,346,2,54,164,26,0,-0.068025,0.067135,0.095575,...,0.040786,0.163686,0.168691,0.028457,1.030331,0.175195,1.045120,0.943116,0.201202,0.964339
10441,18,346,2,54,164,26,0,-0.081430,0.064823,0.104081,...,0.066729,0.149495,0.163711,0.026801,1.013765,0.206048,1.034492,1.041735,0.317350,1.089001
10442,18,346,2,54,164,26,0,-0.063070,0.067157,0.092130,...,0.079896,0.170803,0.188566,0.035557,0.986757,0.157275,0.999212,0.962485,0.266562,0.998716
10443,18,346,2,54,164,26,0,-0.067892,0.079594,0.104616,...,0.058163,0.187315,0.196137,0.038470,1.039934,0.177557,1.054983,0.951916,0.269101,0.989222


In [7]:
features_val

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,19,11,0,78,164,28,0,-0.234197,0.076135,0.246262,...,-0.033745,0.275390,0.277450,0.076978,0.949084,0.171723,0.964494,0.417878,0.133588,0.438711
1,19,11,0,78,164,28,0,-0.288951,0.096326,0.304584,...,0.013696,0.457956,0.458160,0.209911,1.039806,0.283088,1.077652,0.906527,0.852564,1.244450
2,19,11,0,78,164,28,0,-0.270631,0.145508,0.307269,...,0.122430,0.350064,0.370856,0.137534,1.001694,0.227473,1.027197,1.014761,0.654807,1.207689
3,19,11,0,78,164,28,0,-0.268421,0.050643,0.273156,...,-0.006305,0.240823,0.240906,0.058036,1.019236,0.291428,1.060082,0.377032,0.097031,0.389318
4,19,11,0,78,164,28,0,-0.284966,0.069832,0.293398,...,0.054867,0.291726,0.296841,0.088114,1.002974,0.282780,1.042076,0.443488,0.197407,0.485440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,21,350,2,52,165,24,1,-0.278689,0.080730,0.290146,...,0.123326,0.296616,0.321232,0.103190,1.047386,0.158408,1.059297,1.581893,0.754236,1.752500
1941,21,350,2,52,165,24,1,-0.257495,0.090124,0.272811,...,0.185220,0.270118,0.327521,0.107270,0.974041,0.215425,0.997579,1.476576,0.839732,1.698654
1942,21,350,2,52,165,24,1,-0.313846,0.098671,0.328992,...,0.137498,0.376403,0.400730,0.160585,1.054926,0.248217,1.083735,1.732480,0.661034,1.854306
1943,21,350,2,52,165,24,1,-0.248173,0.070854,0.258089,...,0.159190,0.283895,0.325481,0.105938,1.039778,0.257272,1.071133,1.605642,0.964587,1.873103


In [8]:
features_test

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,22,15,0,100,186,31,1,0.275197,0.087849,0.288878,...,-0.106011,0.476606,0.488254,0.238392,1.019939,0.169206,1.033880,0.862116,0.416152,0.957302
1,22,15,0,100,186,31,1,0.251685,0.122715,0.280008,...,-0.004438,0.956823,0.956833,0.915530,1.092591,0.209134,1.112426,1.591224,1.008768,1.884041
2,22,15,0,100,186,31,1,0.249442,0.093761,0.266482,...,0.467040,0.834408,0.956223,0.914363,1.049734,0.179465,1.064965,1.602838,0.827365,1.803780
3,22,15,0,100,186,31,1,0.304708,0.110842,0.324242,...,-0.152270,0.553434,0.573999,0.329475,1.045603,0.176571,1.060407,1.299853,0.536201,1.406104
4,22,15,0,100,186,31,1,0.285822,0.119090,0.309640,...,0.070039,0.916950,0.919621,0.845702,1.089773,0.182332,1.104921,1.709555,0.715838,1.853376
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,24,353,2,74,173,18,0,-0.192117,0.232299,0.301449,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
1594,24,353,2,74,173,18,0,-0.236467,0.225630,0.326842,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
1595,24,353,2,74,173,18,0,-0.209301,0.203869,0.292180,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
1596,24,353,2,74,173,18,0,-0.247126,0.215143,0.327655,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [9]:
print(f"NaN values in train: {features_train.isna().sum().sum()}")
print(f"Inf values in train: {np.isinf(features_train.select_dtypes(include=[np.number])).sum().sum()}")

NaN values in train: 0
Inf values in train: 0


In [10]:
features_data = features_data.replace([np.inf, -np.inf], np.nan)
features_train = features_train.replace([np.inf, -np.inf], np.nan)
features_val = features_val.replace([np.inf, -np.inf], np.nan)
features_test = features_test.replace([np.inf, -np.inf], np.nan)

In [11]:
features_data

,user_id,set,label,weight,height,age,gender,acc_x_mean,acc_x_std,acc_x_rms,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1,1,0,102,188,46,1,0.715572,0.136997,0.728568,...,0.020860,0.262813,0.263640,0.069506,1.052953,0.121381,1.059927,1.301077,0.771064,1.512396
1,1,1,0,102,188,46,1,0.718558,0.198869,0.745570,...,0.374454,0.494895,0.620594,0.385137,1.012239,0.227629,1.037517,1.850563,0.793093,2.013350
2,1,1,0,102,188,46,1,0.715490,0.161866,0.733571,...,0.083601,0.400974,0.409597,0.167769,1.019074,0.184435,1.035629,1.538961,0.785672,1.727913
3,1,1,0,102,188,46,1,0.788775,0.121680,0.798105,...,0.066443,0.303860,0.311039,0.096746,1.083907,0.157921,1.095351,1.401988,0.910345,1.671616
4,1,1,0,102,188,46,1,0.752540,0.167665,0.770991,...,0.218401,0.621696,0.658942,0.434205,1.008821,0.175300,1.023938,1.738690,0.875107,1.946499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,24,353,2,74,173,18,0,-0.192117,0.232299,0.301449,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
13984,24,353,2,74,173,18,0,-0.236467,0.225630,0.326842,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
13985,24,353,2,74,173,18,0,-0.209301,0.203869,0.292180,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
13986,24,353,2,74,173,18,0,-0.247126,0.215143,0.327655,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [12]:
metadata_cols = ['user_id', 'set', 'label', 'weight', 'height', 'age', 'gender']

In [13]:
feature_cols = [col for col in features_train.columns if col not in metadata_cols]

In [14]:
feature_cols

['acc_x_mean',
 'acc_x_std',
 'acc_x_rms',
 'acc_x_energy',
 'acc_y_mean',
 'acc_y_std',
 'acc_y_rms',
 'acc_y_energy',
 'acc_z_mean',
 'acc_z_std',
 'acc_z_rms',
 'acc_z_energy',
 'gyr_x_mean',
 'gyr_x_std',
 'gyr_x_rms',
 'gyr_x_energy',
 'gyr_y_mean',
 'gyr_y_std',
 'gyr_y_rms',
 'gyr_y_energy',
 'gyr_z_mean',
 'gyr_z_std',
 'gyr_z_rms',
 'gyr_z_energy',
 'acc_mag_mean',
 'acc_mag_std',
 'acc_mag_rms',
 'gyr_mag_mean',
 'gyr_mag_std',
 'gyr_mag_rms']

In [15]:
imputer = SimpleImputer(strategy='median') #Median ít ảnh hưởng bởi outlier.

In [16]:
imputer.fit(features_train[feature_cols])

SimpleImputer(strategy='median')

In [17]:
features_data[feature_cols] = imputer.transform(features_data[feature_cols])
features_train[feature_cols] = imputer.transform(features_train[feature_cols])
features_val[feature_cols] = imputer.transform(features_val[feature_cols])
features_test[feature_cols] = imputer.transform(features_test[feature_cols])

In [18]:
features_data[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,0.715572,0.136997,0.728568,0.530811,0.730248,0.128008,0.741382,0.549648,-0.114740,0.172685,...,0.020860,0.262813,0.263640,0.069506,1.052953,0.121381,1.059927,1.301077,0.771064,1.512396
1,0.718558,0.198869,0.745570,0.555875,0.679562,0.152497,0.696462,0.485059,-0.041184,0.183880,...,0.374454,0.494895,0.620594,0.385137,1.012239,0.227629,1.037517,1.850563,0.793093,2.013350
2,0.715490,0.161866,0.733571,0.538126,0.694970,0.116331,0.704639,0.496517,-0.107442,0.162297,...,0.083601,0.400974,0.409597,0.167769,1.019074,0.184435,1.035629,1.538961,0.785672,1.727913
3,0.788775,0.121680,0.798105,0.636972,0.698498,0.132592,0.710971,0.505480,-0.088305,0.222583,...,0.066443,0.303860,0.311039,0.096746,1.083907,0.157921,1.095351,1.401988,0.910345,1.671616
4,0.752540,0.167665,0.770991,0.594428,0.649048,0.114957,0.659150,0.434478,-0.027742,0.137016,...,0.218401,0.621696,0.658942,0.434205,1.008821,0.175300,1.023938,1.738690,0.875107,1.946499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,-0.192117,0.232299,0.301449,0.090872,1.065065,0.251376,1.094328,1.197554,-0.040070,0.225783,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
13984,-0.236467,0.225630,0.326842,0.106826,1.038689,0.384251,1.107485,1.226524,-0.022727,0.177472,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
13985,-0.209301,0.203869,0.292180,0.085369,1.038013,0.323419,1.087230,1.182070,-0.047367,0.181057,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
13986,-0.247126,0.215143,0.327655,0.107358,1.015640,0.168569,1.029534,1.059940,-0.005894,0.277400,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [19]:
features_train[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,0.715572,0.136997,0.728568,0.530811,0.730248,0.128008,0.741382,0.549648,-0.114740,0.172685,...,0.020860,0.262813,0.263640,0.069506,1.052953,0.121381,1.059927,1.301077,0.771064,1.512396
1,0.718558,0.198869,0.745570,0.555875,0.679562,0.152497,0.696462,0.485059,-0.041184,0.183880,...,0.374454,0.494895,0.620594,0.385137,1.012239,0.227629,1.037517,1.850563,0.793093,2.013350
2,0.715490,0.161866,0.733571,0.538126,0.694970,0.116331,0.704639,0.496517,-0.107442,0.162297,...,0.083601,0.400974,0.409597,0.167769,1.019074,0.184435,1.035629,1.538961,0.785672,1.727913
3,0.788775,0.121680,0.798105,0.636972,0.698498,0.132592,0.710971,0.505480,-0.088305,0.222583,...,0.066443,0.303860,0.311039,0.096746,1.083907,0.157921,1.095351,1.401988,0.910345,1.671616
4,0.752540,0.167665,0.770991,0.594428,0.649048,0.114957,0.659150,0.434478,-0.027742,0.137016,...,0.218401,0.621696,0.658942,0.434205,1.008821,0.175300,1.023938,1.738690,0.875107,1.946499
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10440,-0.068025,0.067135,0.095575,0.009135,1.010759,0.169300,1.024839,1.050295,-0.053481,0.173162,...,0.040786,0.163686,0.168691,0.028457,1.030331,0.175195,1.045120,0.943116,0.201202,0.964339
10441,-0.081430,0.064823,0.104081,0.010833,0.998630,0.206582,1.019773,1.039937,-0.038340,0.133919,...,0.066729,0.149495,0.163711,0.026801,1.013765,0.206048,1.034492,1.041735,0.317350,1.089001
10442,-0.063070,0.067157,0.092130,0.008488,0.970171,0.158145,0.982976,0.966242,-0.012686,0.153408,...,0.079896,0.170803,0.188566,0.035557,0.986757,0.157275,0.999212,0.962485,0.266562,0.998716
10443,-0.067892,0.079594,0.104616,0.010945,1.024836,0.174411,1.039571,1.080708,-0.033746,0.142117,...,0.058163,0.187315,0.196137,0.038470,1.039934,0.177557,1.054983,0.951916,0.269101,0.989222


In [20]:
features_val[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.234197,0.076135,0.246262,0.060645,-0.910038,0.168267,0.925463,0.856483,-0.063739,0.095178,...,-0.033745,0.275390,0.277450,0.076978,0.949084,0.171723,0.964494,0.417878,0.133588,0.438711
1,-0.288951,0.096326,0.304584,0.092772,-0.982975,0.292551,1.025586,1.051827,-0.078786,0.102610,...,0.013696,0.457956,0.458160,0.209911,1.039806,0.283088,1.077652,0.906527,0.852564,1.244450
2,-0.270631,0.145508,0.307269,0.094414,-0.946807,0.206804,0.969129,0.939210,-0.000997,0.146658,...,0.122430,0.350064,0.370856,0.137534,1.001694,0.227473,1.027197,1.014761,0.654807,1.207689
3,-0.268421,0.050643,0.273156,0.074614,-0.972301,0.303039,1.018431,1.037202,-0.082992,0.071197,...,-0.006305,0.240823,0.240906,0.058036,1.019236,0.291428,1.060082,0.377032,0.097031,0.389318
4,-0.284966,0.069832,0.293398,0.086082,-0.949256,0.290508,0.992715,0.985482,-0.098254,0.068585,...,0.054867,0.291726,0.296841,0.088114,1.002974,0.282780,1.042076,0.443488,0.197407,0.485440
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,-0.278689,0.080730,0.290146,0.084185,0.978435,0.175011,0.993964,0.987964,-0.093575,0.202992,...,0.123326,0.296616,0.321232,0.103190,1.047386,0.158408,1.059297,1.581893,0.754236,1.752500
1941,-0.257495,0.090124,0.272811,0.074426,0.892482,0.238679,0.923846,0.853491,-0.146591,0.213911,...,0.185220,0.270118,0.327521,0.107270,0.974041,0.215425,0.997579,1.476576,0.839732,1.698654
1942,-0.313846,0.098671,0.328992,0.108236,0.958839,0.239631,0.988330,0.976796,-0.134990,0.266885,...,0.137498,0.376403,0.400730,0.160585,1.054926,0.248217,1.083735,1.732480,0.661034,1.854306
1943,-0.248173,0.070854,0.258089,0.066610,0.976851,0.257903,1.010323,1.020753,-0.084552,0.229814,...,0.159190,0.283895,0.325481,0.105938,1.039778,0.257272,1.071133,1.605642,0.964587,1.873103


In [21]:
features_test[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,0.275197,0.087849,0.288878,0.083451,0.952115,0.171124,0.967370,0.935806,-0.189299,0.117544,...,-0.106011,0.476606,0.488254,0.238392,1.019939,0.169206,1.033880,0.862116,0.416152,0.957302
1,0.251685,0.122715,0.280008,0.078404,1.040228,0.223967,1.064066,1.132236,-0.121902,0.109509,...,-0.004438,0.956823,0.956833,0.915530,1.092591,0.209134,1.112426,1.591224,1.008768,1.884041
2,0.249442,0.093761,0.266482,0.071013,1.003411,0.173056,1.018225,1.036783,-0.114582,0.115001,...,0.467040,0.834408,0.956223,0.914363,1.049734,0.179465,1.064965,1.602838,0.827365,1.803780
3,0.304708,0.110842,0.324242,0.105133,0.964068,0.199198,0.984432,0.969106,-0.205499,0.089406,...,-0.152270,0.553434,0.573999,0.329475,1.045603,0.176571,1.060407,1.299853,0.536201,1.406104
4,0.285822,0.119090,0.309640,0.095877,1.036314,0.188961,1.053401,1.109654,-0.087464,0.087582,...,0.070039,0.916950,0.919621,0.845702,1.089773,0.182332,1.104921,1.709555,0.715838,1.853376
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,-0.192117,0.232299,0.301449,0.090872,1.065065,0.251376,1.094328,1.197554,-0.040070,0.225783,...,-0.151122,1.406393,1.414489,2.000779,1.131568,0.246099,1.158020,1.936990,0.799708,2.095582
1594,-0.236467,0.225630,0.326842,0.106826,1.038689,0.384251,1.107485,1.226524,-0.022727,0.177472,...,-0.115867,1.386607,1.391439,1.936104,1.094722,0.408589,1.168487,2.022268,0.726566,2.148829
1595,-0.209301,0.203869,0.292180,0.085369,1.038013,0.323419,1.087230,1.182070,-0.047367,0.181057,...,-0.083266,1.334841,1.337435,1.788734,1.087030,0.347608,1.141256,1.984320,0.482324,2.042098
1596,-0.247126,0.215143,0.327655,0.107358,1.015640,0.168569,1.029534,1.059940,-0.005894,0.277400,...,-0.072532,1.301314,1.303334,1.698680,1.098966,0.191196,1.115474,1.908579,0.753518,2.051941


In [22]:
print(f"Number of features to scale: {len(feature_cols)}")

Number of features to scale: 30


In [23]:
scaler = StandardScaler()

In [24]:
scaler.fit(features_train[feature_cols])

StandardScaler()

In [25]:
features_data[feature_cols] = scaler.transform(features_data[feature_cols])
features_train[feature_cols] = scaler.transform(features_train[feature_cols])
features_val[feature_cols] = scaler.transform(features_val[feature_cols])
features_test[feature_cols] = scaler.transform(features_test[feature_cols])

In [26]:
features_data[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1.968759,0.582582,2.139563,2.211165,-0.190391,-0.208210,-0.316685,-0.603884,-0.048272,0.537940,...,0.098670,-0.178548,-0.195639,-0.397130,0.282709,-0.231055,-0.019922,0.484583,0.962534,0.573773
1,1.977866,1.267773,2.223284,2.348651,-0.336058,-0.074502,-0.455569,-0.749976,0.137386,0.637396,...,3.818920,0.421243,0.716087,0.231053,-0.390637,0.358629,-0.264677,1.129602,1.019823,1.112100
2,1.968509,0.857985,2.164199,2.251293,-0.291775,-0.271963,-0.430286,-0.724061,-0.029850,0.445652,...,0.758792,0.178514,0.177161,-0.201562,-0.277602,0.118898,-0.285304,0.763825,1.000524,0.805368
3,2.191978,0.412957,2.481970,2.793503,-0.281638,-0.183178,-0.410710,-0.703787,0.018450,0.981224,...,0.578263,-0.072467,-0.074572,-0.342917,0.794618,-0.028254,0.366980,0.603039,1.324738,0.744871
4,2.081486,0.922212,2.348460,2.560129,-0.423752,-0.279466,-0.570930,-0.864385,0.171315,0.221061,...,2.177050,0.748947,0.814035,0.328711,-0.447169,0.068199,-0.412992,0.998280,1.233101,1.040261
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13983,-0.799066,1.637989,0.036400,-0.202088,0.771842,0.465373,0.774543,0.861611,0.140198,1.009659,...,-1.710789,2.776914,2.743838,3.446580,1.582843,0.461136,1.051458,1.231056,1.037025,1.200466
13984,-0.934303,1.564136,0.161434,-0.114574,0.696039,1.190860,0.815221,0.927137,0.183973,0.580469,...,-1.339859,2.725779,2.684966,3.317860,0.973490,1.362970,1.165782,1.331160,0.846817,1.257686
13985,-0.851464,1.323146,-0.009241,-0.232271,0.694095,0.858719,0.752597,0.826586,0.121779,0.612318,...,-0.996864,2.591996,2.547030,3.024558,0.846265,1.024519,0.868358,1.286615,0.211664,1.142992
13986,-0.966806,1.448000,0.165440,-0.111654,0.629798,0.013251,0.574213,0.550342,0.226459,1.468215,...,-0.883929,2.505350,2.459929,2.845329,1.043679,0.156422,0.586774,1.197705,0.916906,1.153570


In [27]:
features_train[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,1.968759,0.582582,2.139563,2.211165,-0.190391,-0.208210,-0.316685,-0.603884,-0.048272,0.537940,...,0.098670,-0.178548,-0.195639,-0.397130,0.282709,-0.231055,-0.019922,0.484583,0.962534,0.573773
1,1.977866,1.267773,2.223284,2.348651,-0.336058,-0.074502,-0.455569,-0.749976,0.137386,0.637396,...,3.818920,0.421243,0.716087,0.231053,-0.390637,0.358629,-0.264677,1.129602,1.019823,1.112100
2,1.968509,0.857985,2.164199,2.251293,-0.291775,-0.271963,-0.430286,-0.724061,-0.029850,0.445652,...,0.758792,0.178514,0.177161,-0.201562,-0.277602,0.118898,-0.285304,0.763825,1.000524,0.805368
3,2.191978,0.412957,2.481970,2.793503,-0.281638,-0.183178,-0.410710,-0.703787,0.018450,0.981224,...,0.578263,-0.072467,-0.074572,-0.342917,0.794618,-0.028254,0.366980,0.603039,1.324738,0.744871
4,2.081486,0.922212,2.348460,2.560129,-0.423752,-0.279466,-0.570930,-0.864385,0.171315,0.221061,...,2.177050,0.748947,0.814035,0.328711,-0.447169,0.068199,-0.412992,0.998280,1.233101,1.040261
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10440,-0.420670,-0.191103,-0.977343,-0.650451,0.615769,0.017241,0.559698,0.528527,0.106347,0.542178,...,0.308323,-0.434731,-0.438155,-0.478828,-0.091425,0.067614,-0.181645,0.064387,-0.519395,-0.015170
10441,-0.461546,-0.216705,-0.935456,-0.641135,0.580912,0.220799,0.544036,0.505099,0.144565,0.193548,...,0.581271,-0.471408,-0.450874,-0.482123,-0.365401,0.238852,-0.297716,0.180151,-0.217351,0.118791
10442,-0.405559,-0.190855,-0.994306,-0.653998,0.499126,-0.043665,0.430268,0.338408,0.209315,0.366685,...,0.719811,-0.416339,-0.387392,-0.464697,-0.812057,-0.031839,-0.683048,0.087123,-0.349426,0.021771
10443,-0.420265,-0.053128,-0.932821,-0.640522,0.656227,0.045146,0.605246,0.597318,0.156160,0.266375,...,0.491143,-0.373666,-0.368054,-0.458900,0.067388,0.080723,-0.073919,0.074717,-0.342825,0.011569


In [28]:
features_val[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,-0.927381,-0.091437,-0.235348,-0.367895,-4.904412,0.011604,0.252451,0.090143,0.080456,-0.150620,...,-0.475838,-0.146045,-0.160367,-0.382258,-1.435101,0.048344,-1.062241,-0.552170,-0.695227,-0.580011
1,-1.094343,0.132167,0.051836,-0.191667,-5.114028,0.690187,0.562008,0.531992,0.042477,-0.084597,...,0.023295,0.325777,0.301202,-0.117690,0.065268,0.666432,0.173680,0.021436,1.174476,0.285838
2,-1.038479,0.676835,0.065054,-0.182658,-5.010082,0.222011,0.387454,0.277265,0.238820,0.306716,...,1.167314,0.046943,0.078210,-0.261737,-0.565035,0.357761,-0.377396,0.148488,0.660207,0.246334
3,-1.031738,-0.373745,-0.102918,-0.291267,-5.083351,0.747449,0.539886,0.498912,0.031863,-0.363670,...,-0.187140,-0.235379,-0.253706,-0.419959,-0.274909,0.712716,-0.018230,-0.600117,-0.790293,-0.633090
4,-1.082191,-0.161232,-0.003246,-0.228360,-5.017122,0.679032,0.460376,0.381926,-0.006661,-0.386870,...,0.456467,-0.103827,-0.110839,-0.360095,-0.543853,0.664722,-0.214888,-0.522106,-0.529265,-0.529797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1940,-1.063049,-0.040545,-0.019259,-0.238769,0.522875,0.048427,0.464239,0.387541,0.005149,0.807187,...,1.176742,-0.091189,-0.048537,-0.330090,0.190638,-0.025552,-0.026793,0.814221,0.918773,0.831789
1941,-0.998421,0.063490,-0.104618,-0.292301,0.275853,0.396050,0.247450,0.083377,-0.128664,0.904186,...,1.827942,-0.159671,-0.032476,-0.321971,-1.022357,0.290894,-0.700887,0.690594,1.141106,0.773926
1942,-1.170256,0.158140,0.172021,-0.106840,0.466558,0.401247,0.446820,0.362279,-0.099382,1.374798,...,1.325849,0.115012,0.154514,-0.215861,0.315334,0.472896,0.240112,0.990989,0.676402,0.941191
1943,-0.969996,-0.149918,-0.177110,-0.335174,0.518323,0.501009,0.514818,0.461704,0.027923,1.045468,...,1.554078,-0.124065,-0.037687,-0.324622,0.064804,0.523147,0.102476,0.842100,1.465793,0.961390


In [29]:
features_test[feature_cols]

,acc_x_mean,acc_x_std,acc_x_rms,acc_x_energy,acc_y_mean,acc_y_std,acc_y_rms,acc_y_energy,acc_z_mean,acc_z_std,...,gyr_z_mean,gyr_z_std,gyr_z_rms,gyr_z_energy,acc_mag_mean,acc_mag_std,acc_mag_rms,gyr_mag_mean,gyr_mag_std,gyr_mag_rms
0,0.625919,0.038298,-0.025500,-0.242795,0.447233,0.027199,0.382018,0.269564,-0.236459,0.048073,...,-1.236169,0.373978,0.378066,-0.061005,-0.263285,0.034377,-0.304410,-0.030696,0.039583,-0.022732
1,0.554225,0.424411,-0.069181,-0.270477,0.700462,0.315719,0.680978,0.713867,-0.066349,-0.023309,...,-0.167492,1.615047,1.574902,1.286666,0.938244,0.255978,0.553483,0.825175,1.580688,0.973143
2,0.547386,0.103768,-0.135782,-0.311024,0.594654,0.037750,0.539250,0.497963,-0.047873,0.025484,...,4.793040,1.298679,1.573344,1.284344,0.229469,0.091315,0.035103,0.838808,1.108946,0.886895
3,0.715907,0.292923,0.148631,-0.123861,0.481584,0.180485,0.434769,0.344887,-0.277350,-0.201898,...,-1.722870,0.572531,0.597076,0.120273,0.161143,0.075252,-0.014678,0.483146,0.351772,0.459552
4,0.658319,0.384274,0.076730,-0.174633,0.689214,0.124592,0.648005,0.662789,0.020575,-0.218107,...,0.616100,1.511999,1.479855,1.147691,0.891640,0.107229,0.471511,0.964078,0.818920,0.940191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,-0.799066,1.637989,0.036400,-0.202088,0.771842,0.465373,0.774543,0.861611,0.140198,1.009659,...,-1.710789,2.776914,2.743838,3.446580,1.582843,0.461136,1.051458,1.231056,1.037025,1.200466
1594,-0.934303,1.564136,0.161434,-0.114574,0.696039,1.190860,0.815221,0.927137,0.183973,0.580469,...,-1.339859,2.725779,2.684966,3.317860,0.973490,1.362970,1.165782,1.331160,0.846817,1.257686
1595,-0.851464,1.323146,-0.009241,-0.232271,0.694095,0.858719,0.752597,0.826586,0.121779,0.612318,...,-0.996864,2.591996,2.547030,3.024558,0.846265,1.024519,0.868358,1.286615,0.211664,1.142992
1596,-0.966806,1.448000,0.165440,-0.111654,0.629798,0.013251,0.574213,0.550342,0.226459,1.468215,...,-0.883929,2.505350,2.459929,2.845329,1.043679,0.156422,0.586774,1.197705,0.916906,1.153570


In [30]:
assert features_train.isna().sum().sum() == 0, "Training data contains NaN!"
assert features_val.isna().sum().sum() == 0, "Validation data contains NaN!"
assert features_test.isna().sum().sum() == 0, "Test data contains NaN!"

assert np.isinf(features_train.select_dtypes(include=[np.number])).sum().sum() == 0, "Training data contains infinity!"
assert np.isinf(features_val.select_dtypes(include=[np.number])).sum().sum() == 0, "Validation data contains infinity!"
assert np.isinf(features_test.select_dtypes(include=[np.number])).sum().sum() == 0, "Test data contains infinity!"

In [31]:
print("\nLabel distribution:")
print(f"Train: {features_train['label'].value_counts().sort_index()}")
print(f"Val: {features_val['label'].value_counts().sort_index()}")
print(f"Test: {features_test['label'].value_counts().sort_index()}")


Label distribution:
Train: label
0     966
1    1135
2    2549
3     976
4    2609
5    2210
Name: count, dtype: int64
Val: label
0    184
1    229
2    467
3    189
4    413
5    463
Name: count, dtype: int64
Test: label
0    139
1    184
2    398
3    156
4    349
5    372
Name: count, dtype: int64


In [32]:
#Summary
print(f"Total features created: {len(feature_cols)}")
print(f"Window size: {WINDOW_SIZE} samples (2 seconds)")
print(f"\nDataset sizes:")
print(f"  Train: {features_train.shape[0]} samples")
print(f"  Val:   {features_val.shape[0]} samples")
print(f"  Test:  {features_test.shape[0]} samples")
print(f"  Total: {features_data.shape[0]} samples")

Total features created: 30
Window size: 10 samples (2 seconds)

Dataset sizes:
  Train: 10445 samples
  Val:   1945 samples
  Test:  1598 samples
  Total: 13988 samples
